In [1]:
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")          # reads ANTHROPIC_API_KEY from .env into the environment
client = anthropic.Anthropic()  # finds the key automatically; it never appears in your code

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)
print(response.content[0].text)

Hello there, friend. How are you?


In [5]:
SYSTEM = """You answer questions about board and card game rules, using ONLY the rulebook excerpts provided.

Rules:
1. Use only the excerpts. Do not use outside knowledge, even if you know the game well.
2. Each excerpt is labeled with its game. Only use excerpts from the game the question asks about.
   If the question names no game, answer separately for each game the excerpts cover, labeled by game.
   If the question uses a term that belongs to only one game (e.g. "hole cards" for Texas Hold'em), treat it as about that game.
3. After each claim, cite the excerpt id in square brackets, e.g. [catan-p9-c0].
4. If the excerpts do not contain the answer, reply exactly: "The provided rules don't cover this."
   You may then mention the closest related rule, clearly labeled as related.
5. Be short and direct."""

In [29]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":   # only move up if we're still in notebooks/
    os.chdir("..")
sys.path.append("src")
print("Working in:", os.getcwd())    # should end in /Game
from rulebook_rag.retrieve import Retriever

retriever = Retriever()

def format_context(chunks):
    parts = []
    for c in chunks:
        parts.append(f"<excerpt id=\"{c['id']}\" game=\"{c['game']}\" page=\"{c['page']}\">\n{c['text']}\n</excerpt>")
    return "\n\n".join(parts)

def answer(question, k=5):
    chunks = retriever.search(question, k=k)
    user_message = f"Excerpts:\n{format_context(chunks)}\n\nQuestion: {question}"
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=2000,                      # was 400
        system=SYSTEM,
        messages=[{"role": "user", "content": user_message}],
    )
    if response.stop_reason == "max_tokens":
        print("WARNING: response was cut off")
    text = "".join(b.text for b in response.content if b.type == "text")
    return text, chunks

Working in: /Users/yasasvikotagiri/Desktop/Game


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13732.52it/s]


In [30]:
for q in [
    "How many private hole cards is each player dealt?",           # direct lookup
    "Do you collect money for landing on Free Parking?",           # myth-buster
    "What penalty does the CATAN rulebook impose for flipping the board?",  # unanswerable
]:
    text, chunks = answer(q)
    print("Q:", q)
    print("A:", text)
    print("Chunks:", [c["id"] for c in chunks])
    print("=" * 60)

Q: How many private hole cards is each player dealt?
A: Each player is dealt two private hole cards in Texas Hold'em. [texas_holdem-p2-c1] The deal is also described as two cards per player, "for their eyes only." [texas_holdem-p2-c0]
Chunks: ['texas_holdem-p2-c1', 'rummy-p4-c1', 'texas_holdem-p2-c0', 'poker-p6-c1', 'poker-p1-c2']
Q: Do you collect money for landing on Free Parking?
A: No. Landing on Free Parking gives you no bonus, and "Relax! Nothing happens." [monopoly-p1-c0] [monopoly-p2-c5]
Chunks: ['monopoly-p1-c0', 'monopoly-p2-c5', 'monopoly-p2-c3', 'monopoly-p2-c0', 'monopoly-p2-c6']
Q: What penalty does the CATAN rulebook impose for flipping the board?
A: The provided rules don't cover this.

Related (not about flipping the board): the CATAN turn structure and dice rolling are described in [catan-p6-c0].
Chunks: ['catan-p11-c0', 'catan-p6-c0', 'texas_holdem-p3-c2', 'texas_holdem-p3-c3', 'rummy-p6-c2']


In [31]:
import csv

with open("eval/questions.csv") as f:
    questions = list(csv.DictReader(f))

results = []
for row in questions:
    text, chunks = answer(row["question"])
    results.append({
        "id": row["id"],
        "question_type": row["question_type"],
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "answer": text,
        "chunks": ";".join(c["id"] for c in chunks),
    })
    print(row["id"], "done")

with open("eval/results_v4.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print("Saved", len(results), "results")

1 done
2 done
3 done
4 done
5 done
6 done
7 done
8 done
9 done
10 done
11 done
12 done
13 done
14 done
15 done
16 done
17 done
18 done
19 done
20 done
21 done
22 done
23 done
24 done
25 done
26 done
27 done
28 done
29 done
30 done
31 done
32 done
33 done
34 done
35 done
36 done
37 done
38 done
39 done
40 done
41 done
Saved 41 results


In [32]:
for r in results:
    if r["question_type"] in ("unanswerable", "myth_buster"):
        print(f"[{r['id']}] {r['question_type']}")
        print("Q:       ", r["question"])
        print("EXPECTED:", r["expected_answer"])
        print("GOT:     ", r["answer"])
        print("-" * 60)

[3] myth_buster
Q:        Is every development card forbidden from being used on the turn you buy it?
EXPECTED: No. Victory Point cards bought that turn may be revealed if they give you enough points to win.
GOT:      No. Development cards generally can't be played on the turn you build them, [catan-p9-c0] but Victory Point cards are an exception. You may play multiple VP cards, even on the turn you buy them, in order to win the game. [catan-p3-c1]
------------------------------------------------------------
[5] unanswerable
Q:        What penalty does the CATAN rulebook impose for flipping over the board?
EXPECTED: Not specified in the supplied rulebook.
GOT:      The provided rules don't cover this.

Related (CATAN turn structure, not a penalty): the turn has two phases, production then action, and play passes clockwise [catan-p6-c0].
------------------------------------------------------------
[9] myth_buster
Q:        Do you collect money for landing on Free Parking?
EXPECTED: No. 

In [33]:
by_id = {c["id"]: c for c in retriever.chunks}
print(by_id["texas_holdem-p2-c2"]["text"])

In Hold'em, a marker called ‘the button' or ‘the dealer button' indicates which player is the dealer for the current game. Before the game begins, the player immediately clockwise from the button posts the "small blind", the first forced bet. The player immediately clockwise from the small blind posts the "big blind", which is typically twice the size of the small blind, but the blinds can vary depending on the stakes and betting structure being played.

In Limit games, the big blind is the same as the small bet, and the small blind is typically half the size of the big blind but may be larger depending on the stakes. For example, in a 4 Limit 2/ game the small blind is $1 and the big blind is $2. In a 30 Limit game, the small blind is $10 15/ and the big blind is $15.


In [34]:
print("1:1" in by_id["blackjack-p2-c0"]["text"])

True


## Faithfulness v2

- Unanswerable: 6/6 correct refusals
- Myth-buster: 6/6 correct, all cited
- Related-rule tails: checked rows 22 and 36 against the chunks — grounded
- New extraction bug: Hold'em "$15/$30 Limit" garbled to "30 Limit ... $10 15/"

In [35]:
JUDGE_SYSTEM = """You grade answers to rulebook questions.
Compare the ANSWER with the EXPECTED answer.
- PASS if the answer states the same facts as EXPECTED (wording may differ; extra correct detail is fine).
- FAIL if it contradicts EXPECTED, misses a key fact, or invents a rule.
- If EXPECTED says the rules don't specify it, PASS only if the answer clearly says the rules don't cover it.
Reply in exactly this format:
VERDICT: PASS or FAIL
REASON: one sentence"""

def judge(question, expected, answer_text):
    msg = f"QUESTION: {question}\nEXPECTED: {expected}\nANSWER: {answer_text}"
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=2000,
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": msg}],
    )

    if response.stop_reason == "max_tokens":
        print("WARNING: response was cut off")
    out = "".join(b.text for b in response.content if b.type == "text")
    if "VERDICT: PASS" in out:
        verdict = "PASS"
    elif "VERDICT: FAIL" in out:
        verdict = "FAIL"
    else:
        verdict = "ERROR"     # the judge didn't answer, so we don't know
    return verdict, out


for r in results:
    r["verdict"], r["judge_note"] = judge(r["question"], r["expected_answer"], r["answer"])
    print(r["id"], r["question_type"], r["verdict"])

1 direct_lookup PASS
2 multi_hop PASS
3 myth_buster PASS
4 multi_hop PASS
5 unanswerable PASS
6 table_figure FAIL
7 direct_lookup PASS
8 multi_hop FAIL
9 myth_buster PASS
10 direct_lookup PASS
11 unanswerable PASS
12 table_figure FAIL
13 direct_lookup PASS
14 multi_hop PASS
15 myth_buster PASS
16 disambiguation PASS
17 unanswerable PASS
18 table_figure FAIL
19 direct_lookup PASS
20 direct_lookup PASS
21 multi_hop FAIL
22 myth_buster PASS
23 disambiguation PASS
24 unanswerable PASS
25 direct_lookup PASS
26 direct_lookup PASS
27 multi_hop PASS
28 myth_buster PASS
29 disambiguation PASS
30 unanswerable PASS
31 direct_lookup PASS
32 direct_lookup PASS
33 multi_hop PASS
34 myth_buster PASS
35 disambiguation PASS
36 unanswerable PASS
37 disambiguation FAIL
38 disambiguation FAIL
39 disambiguation FAIL
40 disambiguation FAIL
41 disambiguation FAIL


In [36]:
for r in results:
    if r["verdict"] == "FAIL":
        print(f"[{r['id']}] {r['question_type']}")
        print("Q:       ", r["question"])
        print("EXPECTED:", r["expected_answer"])
        print("GOT:     ", r["answer"])
        print("JUDGE:   ", r["judge_note"])
        print("-" * 60)

[6] table_figure
Q:        According to the pictured building-cost diagram, what does a development card cost?
EXPECTED: One wool, one wheat, and one ore.
GOT:      The provided rules don't cover this. The excerpts don't include the building-cost diagram or list a resource cost for development cards.

Related rule: when you build a development card, you draw the top card of the deck [catan-p9-c0]. The cost to build a development card in the cost diagram isn't given in the text.
JUDGE:    VERDICT: FAIL
REASON: The expected answer states a development card costs one wool, one wheat, and one ore (from the pictured diagram), but the answer says the rules don't cover it and gives no cost, so it misses the key fact.
------------------------------------------------------------
[8] multi_hop
Q:        If you own a complete colour set while in Jail, may you buy Houses? What building restrictions still apply?
EXPECTED: Yes. You must pay the Title Deed cost and build evenly; you cannot build if a

In [37]:
   print([r["id"] for r in results if not r["answer"].strip()])

[]


In [38]:
   from collections import Counter
   total = Counter(r["question_type"] for r in results)
   passed = Counter(r["question_type"] for r in results if r["verdict"] == "PASS")
   for t in total:
       print(f"{t}: {passed[t]}/{total[t]}")

direct_lookup: 10/10
multi_hop: 5/7
myth_buster: 6/6
unanswerable: 6/6
table_figure: 0/3
disambiguation: 4/9


In [40]:
with open("eval/results_v4.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)